# Toy Transformer: Probabilistic Finite-State Machine

This notebook explores the structural properties of a small-scale Transformer where $V=5$ and $L=4$. We will implement a specific rule: **If position 1 is token 'a', output 'c'**.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# --- 1. Hyperparameters ---
V = 5           # Vocabulary size
L = 4           # Context length
d_model = V + L # Dimension d = V + L = 9

token_to_idx = {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4}
idx_to_token = {v: k for k, v in token_to_idx.items()}

## 2. Input Matrix Construction

We use the **basis-vector trick**: $x(t,p) = e_t + e_{V+p}$. This represents a token's identity in the first $V$ dimensions and its position in the remaining $L$ dimensions.

In [ ]:
def get_input_matrix(tokens):
    token_indices = torch.tensor([token_to_idx[t] for t in tokens])
    content_eb = F.one_hot(token_indices, num_classes=V).float()
    pos_indices = torch.arange(L)
    pos_eb = F.one_hot(pos_indices, num_classes=L).float()
    return torch.cat([content_eb, pos_eb], dim=-1)

# Sequence: [b, a, d, e] -> Position 1 (index 1) is 'a'
input_tokens = ['b', 'a', 'd', 'e']
X = get_input_matrix(input_tokens)

## 3. Hard-Coding the Rule

To satisfy the rule, we configure the weights so the **Query** at the last position looks for **Position 1** in the **Keys**. The **Value** matrix then maps the token 'a' to 'c'.

In [ ]:
W_Q = torch.zeros(d_model, d_model)
W_K = torch.zeros(d_model, d_model)
W_V = torch.zeros(d_model, V)

# Query/Key: High alignment at Position 1 (index V+1)
W_Q[V+1, V+1] = 10.0 
W_K[V+1, V+1] = 10.0

# Value: Map token 'a' (index 0) to 'c' (index 2)
W_V[0, 2] = 1.0

q_layer = torch.nn.Linear(d_model, d_model, bias=False)
k_layer = torch.nn.Linear(d_model, d_model, bias=False)
v_layer = torch.nn.Linear(d_model, V, bias=False)

with torch.no_grad():
    q_layer.weight.copy_(W_Q.T)
    k_layer.weight.copy_(W_K.T)
    v_layer.weight.copy_(W_V.T)

## 4. Attention Execution & Visualization

We calculate the Attention Probability Matrix $P = \text{softmax}(QK^T)$. In our finite-state machine interpretation, $P$ acts as a **probabilistic pointer**.

In [ ]:
Q, K, V_mat = q_layer(X), k_layer(X), v_layer(X)
print("Q:\n", Q)
print("K:\n", K)
print("V_mat:\n", V_mat)

In [ ]:

S = torch.matmul(Q, K.transpose(-2, -1))
print("S:\n", S)

mask = torch.tril(torch.ones(L, L))
S_masked = S.masked_fill(mask == 0, float('-inf'))

P = F.softmax(S_masked, dim=-1)
H = torch.matmul(P, V_mat)

print(f"Output distribution for last token (Index 2 is 'c'):\n{torch.round(H[-1], decimals=3)}")

plt.imshow(P.detach().numpy(), cmap='Blues')
plt.title("Attention Matrix P (Probabilistic Pointer)")
plt.colorbar()
plt.show()